# RSNA Knee MRI — Weak Labeling with Qwen3-8B

This notebook uses **Qwen3-8B** as a zero-shot structured information extractor to weak-label knee MRI reports for 12 target abnormalities (ACL, MCL, menisci, OA compartments, effusion, synovitis, Baker's, contusion, fracture).

For each report, the model returns per-target `label`, `confidence`, `mentioned`, and `negated` fields, which are validated against the ground-truth columns already present in `train.csv`.

**Flow:** load data → load model → build a strict prompt + parser → sanity-check on one report → run over all studies and compare against ground truth.

## 1. Setup & Load Data

Define the 12 target labels and load the competition training CSV.

In [1]:
import pandas as pd
import torch

DATA_DIR = "/kaggle/input/competitions/rsna-knee-abnormality-detection"

train = pd.read_csv("train.csv")

LABELS = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture",
]

labeled = train[train[LABELS].notna().all(axis=1)].copy()

print("Train:", train.shape)
print("Fully labeled:", len(labeled))
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


/environment/miniconda3/lib/python3.11/site-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


Train: (4407, 14)
Fully labeled: 58
CUDA: True
GPU: NVIDIA GeForce RTX 4090


## 2. GPU Memory Cleanup

Utility cell to free VRAM between runs — drops the model/tokenizer/tensors if they exist and reports current memory usage. Safe to re-run at any point, e.g. before reloading the model.

In [2]:
import gc
import torch

for name in [
    "model",
    "tokenizer",
    "inputs",
    "outputs",
    "generated_tokens",
]:
    if name in globals():
        del globals()[name]

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

    allocated = torch.cuda.memory_allocated(0) / 1024**3
    reserved = torch.cuda.memory_reserved(0) / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3

    print(f"Total VRAM:     {total:.2f} GB")
    print(f"Allocated VRAM: {allocated:.2f} GB")
    print(f"Reserved VRAM:  {reserved:.2f} GB")
    print(f"Free-ish VRAM:  {total - reserved:.2f} GB")
else:
    print("CUDA not available.")


GPU: NVIDIA GeForce RTX 4090
Total VRAM:     23.52 GB
Allocated VRAM: 0.00 GB
Reserved VRAM:  0.00 GB
Free-ish VRAM:  23.52 GB


## 3. Model Path

Local path to the cached Qwen3-8B weights.

In [3]:
MODEL_DIR = "/home/featurize/.cache/kagglehub/models/manojkumarcs28/qwen3-8b/pyTorch/qwen3-8b/1/Qwen 3-8B"

print(MODEL_DIR)

/home/featurize/.cache/kagglehub/models/manojkumarcs28/qwen3-8b/pyTorch/qwen3-8b/1/Qwen 3-8B


## 4. Load Qwen3-8B

Load the tokenizer and model with automatic dtype and device placement.

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_PATH = MODEL_DIR

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype="auto",
    device_map="auto",
)

print("Model loaded successfully!")

/environment/miniconda3/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 399/399 [00:02<00:00, 177.47it/s]


Model loaded successfully!


## 5. Verify Device Placement

Confirm how model parameters are distributed across devices (useful when `device_map="auto"` shards across GPU/CPU).

In [5]:
devices = {}

for name, param in model.named_parameters():
    device = str(param.device)
    devices[device] = devices.get(device, 0) + param.numel()

for device, n in devices.items():
    print(device, f"{n/1e9:.2f}B parameters")

from collections import Counter
print(Counter(str(p.device) for p in model.parameters()))

cuda:0 8.19B parameters
Counter({'cuda:0': 399})


## 6. Select Fully-Labeled Studies

Re-derives `labeled` (studies with all 12 ground-truth columns present) and previews a few rows alongside their reports — useful to re-run after a kernel restart without rereading the CSV.

In [6]:
# Studies where ALL 12 ground-truth labels are available
labeled = train[train[LABELS].notna().all(axis=1)].copy()

print("Fully labeled studies:", len(labeled))
print("Unlabeled studies:", train[LABELS].isna().all(axis=1).sum())

display(labeled[["StudyInstanceUID", "Report"] + LABELS].head())

Fully labeled studies: 58
Unlabeled studies: 4349


,StudyInstanceUID,Report,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
29,1.2.826.0.1.3680043.8.498.10095687747295410396...,Antecedentes Clínicos:\nEsguince rodilla. [DAT...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0
63,1.2.826.0.1.3680043.8.498.10170898615867673028...,The study reveals normal knee joint alignment...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
109,1.2.826.0.1.3680043.8.498.10306159113324811538...,Exam Type: MRI KNEE RIGHT WO CONTRAST\nExam Da...,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
445,1.2.826.0.1.3680043.8.498.11287937729196958426...,"MRI of left Knee with \n-3-Plane Loc R'T, Sag ...",0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
470,1.2.826.0.1.3680043.8.498.11382021393803389951...,"In the medial compartment, there is longitudi...",1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 7. Output Parser

Strictly parses and validates the model's JSON response: checks that all 12 labels are present with exactly the right fields and value types, and enforces label–mentioned–negated logical consistency (e.g. `negated=1` requires `mentioned=1`, `mentioned=0` cannot have `label=1`). Raises with the raw output on any violation so failures are easy to debug.

In [7]:
import json
import math

def parse_qwen_output(answer):
    """
    Strictly parse Qwen output.

    Expected format:

    {
        "ACL": {
            "label": 0,
            "confidence": 0.98,
            "mentioned": 1,
            "negated": 1
        },
        ...
    }
    """

    if not isinstance(answer, str):
        raise ValueError(
            f"Qwen output must be a string, got {type(answer).__name__}"
        )

    if "</think>" in answer:
        answer = answer.split("</think>", 1)[1]

    answer = answer.strip()

    start = answer.find("{")

    if start == -1:
        raise ValueError(
            f"No JSON object found in model output:\n{answer}"
        )

    decoder = json.JSONDecoder()

    try:
        obj, end = decoder.raw_decode(answer[start:])
    except json.JSONDecodeError as e:
        raise ValueError(
            f"Invalid JSON from Qwen: {e}\n\n"
            f"RAW OUTPUT:\n{answer}"
        ) from e

    if not isinstance(obj, dict):
        raise ValueError(
            f"Qwen output must be a JSON object, "
            f"got {type(obj).__name__}"
        )

    missing = [
        label for label in LABELS
        if label not in obj
    ]

    unexpected = [
        key for key in obj
        if key not in LABELS
    ]

    if missing:
        raise ValueError(
            f"Missing labels: {missing}\n\n"
            f"RAW OUTPUT:\n{answer}"
        )

    if unexpected:
        raise ValueError(
            f"Unexpected labels: {unexpected}\n\n"
            f"RAW OUTPUT:\n{answer}"
        )

    result = {}

    for label in LABELS:
        item = obj[label]

        if not isinstance(item, dict):
            raise ValueError(
                f"{label}: expected object, "
                f"got {type(item).__name__}"
            )

        required = {
            "label",
            "confidence",
            "mentioned",
            "negated",
        }

        missing_fields = required - set(item)
        unexpected_fields = set(item) - required

        if missing_fields:
            raise ValueError(
                f"{label}: missing fields "
                f"{sorted(missing_fields)}"
            )

        if unexpected_fields:
            raise ValueError(
                f"{label}: unexpected fields "
                f"{sorted(unexpected_fields)}"
            )

        model_label = item["label"]
        confidence = item["confidence"]
        mentioned = item["mentioned"]
        negated = item["negated"]

        # Validate label
        if isinstance(model_label, bool):
            raise ValueError(
                f"{label}: label cannot be boolean"
            )

        if not isinstance(model_label, int):
            raise ValueError(
                f"{label}: label must be an integer"
            )

        if model_label not in (0, 1):
            raise ValueError(
                f"{label}: label must be 0 or 1, "
                f"got {model_label}"
            )

        # Validate confidence
        if isinstance(confidence, bool):
            raise ValueError(
                f"{label}: confidence cannot be boolean"
            )

        if not isinstance(confidence, (int, float)):
            raise ValueError(
                f"{label}: confidence must be numeric"
            )

        confidence = float(confidence)

        if not math.isfinite(confidence):
            raise ValueError(
                f"{label}: confidence must be finite"
            )

        if not 0.0 <= confidence <= 1.0:
            raise ValueError(
                f"{label}: confidence {confidence} "
                f"outside [0, 1]"
            )

        # Validate mentioned
        if isinstance(mentioned, bool):
            raise ValueError(
                f"{label}: mentioned cannot be boolean"
            )

        if not isinstance(mentioned, int):
            raise ValueError(
                f"{label}: mentioned must be an integer"
            )

        if mentioned not in (0, 1):
            raise ValueError(
                f"{label}: mentioned must be 0 or 1"
            )

        # Validate negated
        if isinstance(negated, bool):
            raise ValueError(
                f"{label}: negated cannot be boolean"
            )

        if not isinstance(negated, int):
            raise ValueError(
                f"{label}: negated must be an integer"
            )

        if negated not in (0, 1):
            raise ValueError(
                f"{label}: negated must be 0 or 1"
            )

        # Logical consistency checks
        if mentioned == 0 and negated == 1:
            raise ValueError(
                f"{label}: negated=1 requires mentioned=1"
            )

        if negated == 1 and model_label != 0:
            raise ValueError(
                f"{label}: negated=1 requires label=0"
            )

        if mentioned == 0 and model_label == 1:
            raise ValueError(
                f"{label}: mentioned=0 cannot have label=1"
            )

        result[label] = {
            "label": model_label,
            "confidence": confidence,
            "mentioned": mentioned,
            "negated": negated,
        }

    return result

## 8. Prompting & Inference Function

`qwen_score_report` builds a detailed system + user prompt with per-target definitions, negation rules, and confidence bands, runs greedy generation (`do_sample=False`) with thinking disabled for short, predictable output, then parses the JSON response through `parse_qwen_output`.

In [8]:
import json
import torch

def qwen_score_report(report, max_new_tokens=500):
    """
    Classify one knee MRI report into the 12 predefined abnormalities.

    For each target:
        label: 0 = absent, 1 = present
        confidence: 0.0-1.0 confidence in the classification
        mentioned: 0 = not mentioned, 1 = mentioned
        negated: 0 = not explicitly negated, 1 = explicitly negated

    Thinking is disabled to keep the output short and predictable.
    """

    output_template = {
        label: {
            "label": 0,
            "confidence": 0.50,
            "mentioned": 0,
            "negated": 0
        }
        for label in LABELS
    }

    prompt = f"""
You are an expert musculoskeletal radiologist performing structured information extraction from a knee MRI report.

Read the complete MRI report and classify ALL 12 targets independently.

Use ONLY information explicitly stated in the MRI report.

Do not use:
- clinical history
- symptoms
- mechanism of injury
- age
- treatment
- outside information to invent findings

CLASSIFICATION

For every target determine:

label:
0 = abnormality is absent
1 = abnormality is present

mentioned:
0 = the report does not provide information about this specific finding
1 = the report explicitly discusses the relevant structure or finding

negated:
0 = the finding is not explicitly negated
1 = the finding is explicitly described as absent, normal, intact, preserved, or without the relevant pathology

IMPORTANT:
Not mentioned = label 0, mentioned 0, negated 0.

Do NOT interpret an unmentioned finding as a confirmed negative.

If a finding is explicitly normal or intact:
label = 0
mentioned = 1
negated = 1

If a finding is explicitly present:
label = 1
mentioned = 1
negated = 0

If wording is uncertain, such as "possible", "suspicious for", "cannot exclude", or "questionable", classify according to the evidence actually stated and lower confidence.

Be conservative about positive classifications. Do not call pathology present merely because it could be inferred from another finding.

CONFIDENCE

confidence is a number from 0.0 to 1.0 representing confidence in the classification.

Use approximately:

0.95-1.00 = explicit and unambiguous
0.85-0.94 = strong evidence
0.70-0.84 = moderate evidence
0.50-0.69 = ambiguous, equivocal, or weak evidence
0.00-0.49 = very uncertain

Confidence is NOT disease probability.

TARGET DEFINITIONS

ACL

Abnormality specifically involving the anterior cruciate ligament.

label=1 when the report explicitly describes ACL tear, rupture, discontinuity, abnormal signal, injury, or other ACL pathology.

label=0 when the ACL is explicitly normal, intact, preserved, or without pathology.

MCL

Abnormality specifically involving the medial collateral ligament.

label=1 when the report explicitly describes MCL tear, rupture, sprain, abnormal signal, thickening, injury, or other MCL pathology.

label=0 when the MCL is explicitly normal, intact, preserved, or without pathology.

MEDIAL MENISCUS

Pathology specifically involving the medial meniscus.

label=1 for explicit tear, rupture, truncation, extrusion, displaced fragment, abnormal morphology, or other medial meniscus pathology.

label=0 when the medial meniscus is explicitly normal, preserved, or without tear/pathology.

A lateral meniscus finding does not affect this target.

LATERAL MENISCUS

Pathology specifically involving the lateral meniscus.

label=1 for explicit tear, rupture, truncation, extrusion, displaced fragment, abnormal morphology, or other lateral meniscus pathology.

label=0 when the lateral meniscus is explicitly normal, preserved, or without tear/pathology.

A medial meniscus finding does not affect this target.

MEDIAL OA

Degenerative or chondral pathology specifically involving the medial femorotibial compartment.

label=1 for medial femorotibial cartilage loss, medial chondral damage, medial compartment osteoarthritis, medial degenerative change, medial osteophytes, or equivalent explicit pathology.

label=0 when the medial femorotibial compartment is explicitly normal or without degenerative/chondral pathology.

LATERAL OA

Degenerative or chondral pathology specifically involving the lateral femorotibial compartment.

label=1 for lateral femorotibial cartilage loss, lateral chondral damage, lateral compartment osteoarthritis, lateral degenerative change, lateral osteophytes, or equivalent explicit pathology.

label=0 when the lateral femorotibial compartment is explicitly normal or without degenerative/chondral pathology.

PF OA

Degenerative or chondral pathology specifically involving the patellofemoral compartment.

This includes the patella and trochlea.

label=1 for patellar cartilage damage, trochlear cartilage damage, patellofemoral chondropathy, patellofemoral osteoarthritis, or equivalent explicit pathology.

label=0 when the patellofemoral compartment is explicitly normal or without pathology.

Compartment rule:
Patella/trochlea = PF OA.
Medial femorotibial = Medial OA.
Lateral femorotibial = Lateral OA.

Never transfer pathology between compartments.

EFFUSION

Abnormally increased intra-articular joint fluid.

label=1 for explicit joint effusion or increased intra-articular fluid.

label=0 when the report explicitly states no joint effusion.

Do not infer effusion from pain, injury, synovitis, or other findings.

SYNOVITIS

Inflammation or abnormality of the synovium.

label=1 for explicit synovitis, synovial inflammation, synovial thickening, synovial hypertrophy, or equivalent synovial pathology.

label=0 when the synovium is explicitly normal or without synovial pathology.

Effusion alone does not establish synovitis.

BAKER'S

Baker's cyst or popliteal cyst.

label=1 for Baker's cyst, popliteal cyst, or clearly equivalent cyst.

label=0 when the report explicitly states there is no Baker's or popliteal cyst.

Do not infer Baker's cyst from nonspecific posterior fluid.

CONTUSION

Traumatic bone contusion or bone bruise.

label=1 for explicit bone contusion or bone bruise.

label=0 when the report explicitly states no bone contusion or bone bruise.

Bone marrow edema alone does not establish contusion.

FRACTURE

Bone fracture.

label=1 for explicit fracture, fracture line, cortical disruption, or equivalent fracture finding.

label=0 when the report explicitly states no fracture.

Bone marrow edema alone does not establish fracture.

NEGATION

Negation applies only to the finding actually described.

Examples:

"ACL intact"
ACL: label=0, mentioned=1, negated=1

"menisco medial sin signos de rotura"
Medial Meniscus: label=0, mentioned=1, negated=1

"lateral meniscus tear"
Lateral Meniscus: label=1, mentioned=1, negated=0

"trochlear chondropathy"
PF OA: label=1, mentioned=1, negated=0

"mild joint effusion"
Effusion: label=1, mentioned=1, negated=0

"no joint effusion"
Effusion: label=0, mentioned=1, negated=1

"no bone marrow signal abnormality"
Do not infer Contusion or Fracture.

OUTPUT

Return exactly ONE JSON object.

Do not output reasoning.
Do not output explanations.
Do not output markdown.
Do not output code fences.
Do not output anything before the JSON.
Do not output anything after the JSON.
Do not output additional fields.

The JSON must contain exactly these 12 labels:

{json.dumps(output_template, indent=2)}

Each label must contain exactly these four fields:
"label"
"confidence"
"mentioned"
"negated"

label must be an integer 0 or 1.
confidence must be a number between 0.0 and 1.0.
mentioned must be an integer 0 or 1.
negated must be an integer 0 or 1.

MRI REPORT

{report}

FINAL ANSWER:
"""

    messages = [
        {
            "role": "system",
            "content": (
                "You are an expert musculoskeletal radiologist. "
                "Extract only findings explicitly supported by the MRI report. "
                "Be conservative about positive classifications. "
                "Return only the requested JSON object."
            ),
        },
        {
            "role": "user",
            "content": prompt,
        },
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
    ).to(model.device)

    input_length = inputs["input_ids"].shape[1]

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_tokens = outputs[0][input_length:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    ).strip()

    try:
        prediction = parse_qwen_output(answer)
    except Exception as e:
        raise ValueError(
            "Qwen returned an invalid structured answer.\n\n"
            f"Raw output:\n{answer}\n\n"
            f"Parser error:\n{e}"
        ) from e

    return prediction, answer

## 9. Run Single Test

Sanity-check the pipeline end-to-end on one annotated report: run inference, compare each of the 12 targets against ground truth, and print the raw model output for inspection.

In [9]:
# Test one annotated study

tmp = 0
test_row = labeled.iloc[tmp]

report = test_row["Report"]

prediction, raw_output = qwen_score_report(report)

rows = []

for target in LABELS:
    true_value = int(float(test_row[target]))
    pred = prediction[target]

    rows.append({
        "Target": target,
        "True": true_value,
        "Qwen": pred["label"],
        "Confidence": pred["confidence"],
        "Mentioned": pred["mentioned"],
        "Negated": pred["negated"],
        "Correct": true_value == pred["label"],
    })

comparison = pd.DataFrame(rows)

print("=" * 100)
print("MRI REPORT")
print("=" * 100)
print(report)

print("\n" + "=" * 100)
print("GROUND TRUTH vs QWEN")
print("=" * 100)

display(comparison)

print("\n" + "=" * 100)
print("SUMMARY")
print("=" * 100)

accuracy = comparison["Correct"].mean()

print(f"Accuracy: {accuracy:.1%}")
print(f"Predicted positives: {comparison['Qwen'].sum()}")
print(f"True positives: {comparison['True'].sum()}")
print(f"Explicitly mentioned: {comparison['Mentioned'].sum()}")
print(f"Explicitly negated: {comparison['Negated'].sum()}")

print("\n" + "=" * 100)
print("RAW QWEN OUTPUT")
print("=" * 100)
print(raw_output)

MRI REPORT
Antecedentes Clínicos:
Esguince rodilla. [DATE].
Hallazgos:
No hay alteraciones de señal significativas de la médula ósea.
Ligamentos cruzados y colaterales dentro de límites normales.
Amputación marginal del cuerpo del menisco lateral. Menisco medial de morfología y señal
conservada, sin signos de rotura.
Cartílagos de los compartimentos femorotibiales sin alteraciones.
Fina úlcera condral focal de espesor total del aspecto inferior de la vertiente medial de la tróclea
femoral con mínimos cambios óseos secundarios. Fenómenos condrales reparativos de la región
central de la simple femoral. Cartílago rotuliano sin alteraciones.
Leve derrame articular. No hay quistes poplíteos patológicos.
Aumento de señal de la inserción distal del tendón cuadricipital y de la inserción proximal del
tendón rotuliano, sin signos de rotura.
No hay alteraciones de señal de la grasa de Hoffa.
Impresión:
Amputación marginal del cuerpo del menisco lateral.
Condropatía focal grado 4 del aspecto infe

,Target,True,Qwen,Confidence,Mentioned,Negated,Correct
0,ACL,0,0,1.0,1,1,True
1,MCL,0,0,1.0,1,1,True
2,Medial Meniscus,0,0,1.0,1,1,True
3,Lateral Meniscus,0,1,1.0,1,0,False
4,Medial OA,0,0,1.0,1,1,True
5,Lateral OA,0,0,1.0,1,1,True
6,PF OA,1,1,1.0,1,0,True
7,Effusion,1,1,1.0,1,0,True
8,Synovitis,0,0,0.5,0,0,True
9,Baker's,0,0,1.0,1,1,True



SUMMARY
Accuracy: 91.7%
Predicted positives: 3
True positives: 2
Explicitly mentioned: 10
Explicitly negated: 7

RAW QWEN OUTPUT
{
  "ACL": {
    "label": 0,
    "confidence": 1.0,
    "mentioned": 1,
    "negated": 1
  },
  "MCL": {
    "label": 0,
    "confidence": 1.0,
    "mentioned": 1,
    "negated": 1
  },
  "Medial Meniscus": {
    "label": 0,
    "confidence": 1.0,
    "mentioned": 1,
    "negated": 1
  },
  "Lateral Meniscus": {
    "label": 1,
    "confidence": 1.0,
    "mentioned": 1,
    "negated": 0
  },
  "Medial OA": {
    "label": 0,
    "confidence": 1.0,
    "mentioned": 1,
    "negated": 1
  },
  "Lateral OA": {
    "label": 0,
    "confidence": 1.0,
    "mentioned": 1,
    "negated": 1
  },
  "PF OA": {
    "label": 1,
    "confidence": 1.0,
    "mentioned": 1,
    "negated": 0
  },
  "Effusion": {
    "label": 1,
    "confidence": 1.0,
    "mentioned": 1,
    "negated": 0
  },
  "Synovitis": {
    "label": 0,
    "confidence": 0.5,
    "mentioned": 0,
    "negate

## 10. Weak-label train.csv & Generate Target File

In [ ]:
import pandas as pd
import os
import time
from datetime import datetime

INPUT_FILE = "train.csv"
OUTPUT_FILE = "qwen_knee_targets_4000.csv"
LOG_FILE = "labeling.log"

UID_COLUMN = "StudyInstanceUID"
REPORT_COLUMN = "Report"


def log(message):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{timestamp}] {message}"

    print(line, flush=True)

    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(line + "\n")


log("begin labeling")

train = pd.read_csv(INPUT_FILE)
log(f"Loaded {len(train)} rows from {INPUT_FILE}")

work = train[[UID_COLUMN, REPORT_COLUMN]].copy()
work[UID_COLUMN] = work[UID_COLUMN].astype(str)
work = work.dropna(subset=[REPORT_COLUMN])
work[REPORT_COLUMN] = work[REPORT_COLUMN].astype(str)
work = work.drop_duplicates(subset=[UID_COLUMN]).reset_index(drop=True)

log(f"Unique studies with reports: {len(work)}")

if os.path.exists(OUTPUT_FILE):
    existing = pd.read_csv(OUTPUT_FILE)

    if "status" in existing.columns:
        completed = set(
            existing.loc[
                existing["status"] == "ok",
                UID_COLUMN
            ].astype(str)
        )
    else:
        completed = set(existing[UID_COLUMN].astype(str))

    log(f"Existing rows: {len(existing)}")
    log(f"Successfully completed: {len(completed)}")
else:
    completed = set()
    log("No previous output found.")

todo = work[
    ~work[UID_COLUMN].isin(completed)
].reset_index(drop=True)

log(f"Remaining: {len(todo)}")

if len(todo) == 0:
    log("Nothing left to process.")

else:
    new_rows = []
    total_start = time.time()

    for n, (_, row) in enumerate(todo.iterrows(), start=1):
        uid = row[UID_COLUMN]
        report = row[REPORT_COLUMN]
        start = time.time()

        try:
            prediction, raw_output = qwen_score_report(
                report,
                max_new_tokens=3000
            )

            elapsed = time.time() - start

            result = {
                UID_COLUMN: uid,
                REPORT_COLUMN: report,
                "qwen_raw": raw_output,
                "status": "ok",
                "error": "",
                "generation_seconds": elapsed,
            }

            for target in LABELS:
                pred = prediction[target]

                result[f"{target}__label"] = pred["label"]
                result[f"{target}__confidence"] = pred["confidence"]
                result[f"{target}__mentioned"] = pred["mentioned"]
                result[f"{target}__negated"] = pred["negated"]

            new_rows.append(result)

            log(
                f"[{n}/{len(todo)}] OK {uid[:24]}... "
                f"{elapsed:.1f}s"
            )

        except Exception as e:
            elapsed = time.time() - start

            result = {
                UID_COLUMN: uid,
                REPORT_COLUMN: report,
                "qwen_raw": "",
                "status": "error",
                "error": repr(e),
                "generation_seconds": elapsed,
            }

            for target in LABELS:
                result[f"{target}__label"] = None
                result[f"{target}__confidence"] = None
                result[f"{target}__mentioned"] = None
                result[f"{target}__negated"] = None

            new_rows.append(result)

            log(
                f"[{n}/{len(todo)}] ERROR: {uid}\n"
                f"{type(e).__name__}: {e}"
            )

        if len(new_rows) >= 25 or n == len(todo):
            chunk = pd.DataFrame(new_rows)

            if os.path.exists(OUTPUT_FILE):
                chunk.to_csv(
                    OUTPUT_FILE,
                    mode="a",
                    header=False,
                    index=False
                )
            else:
                chunk.to_csv(
                    OUTPUT_FILE,
                    mode="w",
                    header=True,
                    index=False
                )

            log(
                f"Checkpoint saved: {len(chunk)} rows"
            )

            new_rows = []

        if n % 25 == 0 or n == len(todo):
            elapsed_total = time.time() - total_start
            rate = n / elapsed_total if elapsed_total > 0 else 0
            remaining = len(todo) - n
            eta_min = remaining / rate / 60 if rate > 0 else 0

            log(
                f"Progress: {n}/{len(todo)} "
                f"({100 * n / len(todo):.1f}%) | "
                f"ETA: {eta_min:.1f} min"
            )

    log("\nDONE")
    log(f"Output: {OUTPUT_FILE}")

    final = pd.read_csv(OUTPUT_FILE)
    log(f"Rows: {len(final)}")

    if "status" in final.columns:
        log(f"Successful: {(final['status'] == 'ok').sum()}")
        log(f"Errors: {(final['status'] == 'error').sum()}")

[2026-08-23 10:12:43] begin labeling
[2026-08-23 10:12:43] Loaded 4407 rows from train.csv
[2026-08-23 10:12:43] Unique studies with reports: 4407
[2026-08-23 10:12:43] Existing rows: 375
[2026-08-23 10:12:43] Successfully completed: 375
[2026-08-23 10:12:43] Remaining: 4032
[2026-08-23 10:12:54] [1/4032] OK 1.2.826.0.1.3680043.8.49... 11.1s
[2026-08-23 10:13:05] [2/4032] OK 1.2.826.0.1.3680043.8.49... 11.2s
[2026-08-23 10:13:16] [3/4032] OK 1.2.826.0.1.3680043.8.49... 11.1s
[2026-08-23 10:13:37] [4/4032] OK 1.2.826.0.1.3680043.8.49... 11.1s
[2026-08-23 10:13:48] [5/4032] OK 1.2.826.0.1.3680043.8.49... 11.0s
[2026-08-23 10:13:59] [6/4032] OK 1.2.826.0.1.3680043.8.49... 11.1s
[2026-08-23 10:14:10] [7/4032] OK 1.2.826.0.1.3680043.8.49... 11.4s
[2026-08-23 10:14:22] [8/4032] OK 1.2.826.0.1.3680043.8.49... 11.1s
[2026-08-23 10:14:33] [9/4032] OK 1.2.826.0.1.3680043.8.49... 11.1s
[2026-08-23 10:14:44] [10/4032] OK 1.2.826.0.1.3680043.8.49... 11.3s
[2026-08-23 10:14:55] [11/4032] OK 1.2.826.